# 03 — Cache additions: real C2C12 sequences + Phase A fault set (`cultureQC_upgrade.md` §2A.0 / §2A.3)

**Why this notebook exists.** The Slice 1b cache (`nb/02`) holds synthetic tiles, EVICAN and
AutoQC-Bench — **no time-lapse sequences**. So Slice 2's replay and Slice 3's backtest could only
run on hand-made synthetic curves (`results/growth_backtest.md` says so). Phase A has nothing real to
validate until real sequences are in the cache. This notebook adds them, plus the A3 fault set, in
**one short GPU run** — then everything else in Phase A runs on the Mac.

**What it adds to the existing Drive cache** (same `CACHE_DIR` as `nb/02`; `Cache.build()` skips keys already present):
| Set | ~n images (defaults) | Role |
|---|---:|---|
| C2C12 (24 of 48 sequences, hourly frames) | ~2,000 | normal fleet for replay, growth backtest (V3), density confound (V4), SPC false alarms (V6) |
| `c2c12_fault_contam` (4 sequences, post-onset, every 2 h) | ~90 | contamination-onset detection (V6) |
| `c2c12_fault_dim` (all selected sequences, post-onset, every 2 h) | ~550 | instrument-vs-culture (V7) |
| growth-stall fault | 0 new | metadata only — reuses cached frames (V6) |

Every image gets real `sequence_id`, `frame_idx`, `timestamp` — the fields `culture/replay.py` requires.

**Data licence.** C2C12: Ker et al., *Sci Data* 5:180237 (2018), doi:10.1038/sdata.2018.237 —
OSF `ysaq2`, **CC BY 4.0**. Credit it anywhere these images or derived numbers appear (README, report, demo).

**Three ⏸ checkpoints** — don't "Run all": (1) OSF layout, (2) sequences + sample frames,
(3) budget test before the full GPU pass.

`PINNED_SHA` below pins a commit that has `scripts/fetch_c2c12.py`, `scripts/make_fault_set.py`, `scripts/cache_tile_embeddings.py` and the `Cache.build()` periodic-flush fix.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/cultureqc'
STAGE_DIR = f'{DRIVE_ROOT}/staged'
CACHE_DIR = f'{DRIVE_ROOT}/cache'        # the SAME cache nb/02 built — we add to it
SLIM_DIR = f'{DRIVE_ROOT}/cache_slim'
LOCAL_ROOT = '/content/data'             # inputs on fast local disk (Drive FUSE is slow for many small files)
C2C12_DIR = f'{LOCAL_ROOT}/c2c12'        # fetch_c2c12.py output: raw/, png/, csvs, normalization json
FAULT_DIR = f'{LOCAL_ROOT}/c2c12_faults' # make_fault_set.py output
import os
for d in (STAGE_DIR, LOCAL_ROOT):
    os.makedirs(d, exist_ok=True)
assert os.path.exists(f'{CACHE_DIR}/MANIFEST.json'), 'No existing cache on Drive — run nb/02 first'

In [ ]:
REPO_URL = 'https://github.com/n1tishc/cultureqc.git'
BRANCH = 'slice-1b-compute-cache'
PINNED_SHA = '2f8b5d0'   # nb/03 scripts + Cache.build() periodic flush + frame-number check

!rm -rf /content/cultureqc
!git clone --branch $BRANCH $REPO_URL /content/cultureqc
%cd /content/cultureqc
!git checkout $PINNED_SHA
!git rev-parse HEAD
for f in ('scripts/fetch_c2c12.py', 'scripts/make_fault_set.py', 'scripts/cache_tile_embeddings.py', 'culture/cache.py', 'culture/replay.py'):
    assert os.path.exists(f), f'{f} missing at this commit — commit the new scripts and update PINNED_SHA'

In [ ]:
!pip install -q -r requirements.txt requests
import torch
print('torch', torch.__version__, '— CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU — Runtime > Change runtime type > GPU (needed for step 4)'

In [ ]:
import sys, json, time, glob, random, shutil
sys.path.insert(0, '/content/cultureqc')
sys.path.insert(0, '/content/cultureqc/scripts')
import numpy as np, pandas as pd
import fetch_c2c12 as fc
import make_fault_set as mfs
from culture.cache import Cache, ImageRecord, image_sha256

## 0 — Restore prepared data if a previous run staged it

Steps 1–3 are CPU/network work (~download + convert + faults). They stage their result to Drive as one
archive, so after a disconnect you restore it here and jump straight to step 4.

In [ ]:
PREPARED_ZIP = f'{STAGE_DIR}/c2c12_prepared.zip'
RESTORED = False
if os.path.exists(PREPARED_ZIP):
    !unzip -q -o $PREPARED_ZIP -d /
    RESTORED = os.path.exists(f'{C2C12_DIR}/c2c12_frames.csv') and os.path.exists(f'{FAULT_DIR}/fault_manifest.parquet')
print('restored prepared data from Drive:', RESTORED, '— if True, skip to step 4')

## 1 — Discover how OSF packages the dataset

Walks OSF project `ysaq2` (all storage providers and child components). Frame folders are only *probed*
(first page), not fully listed — `fetch` lists just the sequences it selects. The inventory is saved to
Drive and reused on reruns.

In [ ]:
INVENTORY = f'{STAGE_DIR}/c2c12_osf_inventory.json'
if not RESTORED:
    inv = json.load(open(INVENTORY)) if os.path.exists(INVENTORY) else None
    if inv is None or inv['mode'] != 'tiff_folders':
        # a stale inventory from the pre-fix discover (one component level only) says mode: unknown
        inv = fc.discover(INVENTORY)   # ~8 min: walks all 3 experiments x 19 components
    fc.summarize_inventory(inv)

### ⏸ Checkpoint 1 — read the inventory above

- **`mode: tiff_folders`** → good: loose TIFFs per sequence, the next cells download only every 12th frame.
- **`mode: archives`** → the data ships as archives. Don't run step 2's normal cell; use the
  **archive fallback** cell right after it (downloads a few archives, extracts, then fetches with `--local-src`).
- **`mode: unknown`** → OSF layout isn't what we expected. Download a handful of sequences by hand from
  `https://osf.io/ysaq2/` into `/content/c2c12_src/<sequence folder>/*.tif` and use the fallback cell's
  last line (`LOCAL_SRC = ...`).

Also check the printed sequence folder names: do they state the condition (control / FGF2 / BMP2 / both)?
If not, conditions will read `unknown` — fine for Phase A, but you can supply `CONDITION_MAP_CSV`
(columns `leaf,condition`) from the paper's Table 1 to fix that.

## 2 — Select sequences, download hourly frames, convert to 8-bit PNG

In [ ]:
PER_EXPERIMENT = 8     # 8 x 3 experiments = 24 of 48 sequences (spread evenly within each experiment)
STRIDE = 12            # every 12th 5-min frame = hourly; replay cadences of 6-24 h need no finer
MAX_SEQUENCES = None   # e.g. 6 for a quick dry run
CONDITION_MAP_CSV = None
LOCAL_SRC = None       # set by the archive fallback below if needed

if not RESTORED and LOCAL_SRC is None and inv['mode'] == 'tiff_folders':
    t0 = time.time()
    res = fc.fetch(C2C12_DIR, inventory=INVENTORY, per_experiment=PER_EXPERIMENT, stride=STRIDE,
                   max_sequences=MAX_SEQUENCES, condition_map_csv=CONDITION_MAP_CSV)
    print(f'fetch + convert: {(time.time()-t0)/60:.1f} min')

In [ ]:
# -- ARCHIVE FALLBACK (only if checkpoint 1 showed mode: archives / unknown) --
# Pick archive indices from the printed list; each is downloaded to local disk, extracted, deleted.
ARCHIVE_PICK = []      # e.g. [0, 1, 2] — start small, check sizes printed at checkpoint 1
if not RESTORED and ARCHIVE_PICK:
    SRC = '/content/c2c12_src'
    os.makedirs(SRC, exist_ok=True)
    for i in ARCHIVE_PICK:
        a = inv['archives'][i]
        local = f"/content/{a['name']}"
        print('downloading', a['path'], f"{(a['size'] or 0)/1e9:.2f} GB")
        fc._download(a['download'], local)
        if local.lower().endswith('.zip'):
            !unzip -q -o "$local" -d $SRC
        else:
            !tar -xf "$local" -C $SRC
        os.remove(local)
    LOCAL_SRC = SRC
# LOCAL_SRC = '/content/c2c12_src'   # <- or point at sequences you downloaded by hand

if not RESTORED and LOCAL_SRC:
    res = fc.fetch(C2C12_DIR, local_src=LOCAL_SRC, per_experiment=PER_EXPERIMENT, stride=STRIDE,
                   max_sequences=MAX_SEQUENCES, condition_map_csv=CONDITION_MAP_CSV)

In [ ]:
seqs = pd.read_csv(f'{C2C12_DIR}/c2c12_sequences.csv')
frames = pd.read_csv(f'{C2C12_DIR}/c2c12_frames.csv')
print(json.load(open(f'{C2C12_DIR}/c2c12_normalization.json')))
print(len(seqs), 'sequences |', len(frames), 'frames')
distinct = frames.groupby('sequence_id').frame_idx.nunique()
bad = seqs[seqs.n_frames_kept != seqs.sequence_id.map(distinct)]
assert bad.empty, f'frames collapsed onto repeated frame_idx in: {bad.sequence_id.tolist()}'
print(seqs.groupby(['experiment', 'condition']).size().rename('n_sequences'))
seqs[['sequence_id', 'condition', 'n_frames_total', 'n_frames_kept', 'hours_span']]

In [ ]:
# Visual check: first / middle / last kept frame of 3 sequences, exactly as the cache will read them.
import cv2, matplotlib.pyplot as plt
pick = seqs.sequence_id.iloc[np.linspace(0, len(seqs)-1, 3).astype(int)].tolist()
fig, ax = plt.subplots(3, 3, figsize=(12, 9))
for r, sid in enumerate(pick):
    f = frames[frames.sequence_id == sid].sort_values('frame_idx')
    for c, row in enumerate([f.iloc[0], f.iloc[len(f)//2], f.iloc[-1]]):
        img = cv2.imread(row.png_path, cv2.IMREAD_GRAYSCALE)
        ax[r, c].imshow(img, cmap='gray', vmin=0, vmax=255)
        ax[r, c].set_title(f"{sid[-24:]}  t={row.hours_since_start:.0f}h  mean={img.mean():.0f}", fontsize=8)
        ax[r, c].axis('off')
plt.tight_layout(); plt.show()

### ⏸ Checkpoint 2 — sequences and frames look right?

- Frames should be **mid-grey phase contrast**, not black or blown out (normalization is one fixed linear map
  for the whole dataset, anchored at camera zero — never per-frame, so real brightness changes survive).
- Density should visibly **increase** from first to last frame. That's the growth Phase A fits.
- `hours_span` ≈ 84–89 h per sequence (paper: ~3.5 days).
- The cell above asserts every kept frame has its own `frame_idx` (`fetch_c2c12.py` also refuses to download if the
  frame numbers in a sequence's filenames collide). At checkpoint 1, check in `sample_names` that the frame number
  is the **last** number in each filename — that's what `frame_index()` reads.

## 3 — Build the fault set (CPU)

Contamination onset and lamp dimming create new images; growth stall is metadata only. Bacterial
sprites come from DeepBacs via the repo's own `extract_sprites.py` — regenerated here because `nb/00`
didn't stage them to Drive.

In [ ]:
if not RESTORED:
    if not glob.glob('data/sprites/bacteria/*.png'):
        !python scripts/download_sources.py --out data/sources
        !python scripts/extract_sprites.py --input data/sources/deepbacs --out data/sprites/bacteria
    print(len(glob.glob('data/sprites/bacteria/*.png')), 'bacterial sprites')

In [ ]:
FAULT_CFG = mfs.DEFAULTS   # contamination: 4 seqs, onset at 45% of span, 20->400 sprites/tile-area over 24 h
                           # growth stall: 4 other seqs, onset at 35%, 0.4x time dilation
                           # lamp dimming: all seqs, onset 40 h, 1.0 -> 0.65 over 24 h
print(json.dumps(FAULT_CFG, indent=2))
if not RESTORED:
    t0 = time.time()
    fres = mfs.build(C2C12_DIR, FAULT_DIR, 'data/sprites/bacteria', '/content/cultureqc', FAULT_CFG)
    print(f'fault set: {(time.time()-t0)/60:.1f} min, {len(fres["new_images"])} new images')
man = pd.read_parquet(f'{FAULT_DIR}/fault_manifest.parquet')
man.groupby('fault_type').agg(sequences=('fault_sequence_id', 'nunique'), rows=('frame_idx', 'size'),
                              modified_frames=('is_modified', 'sum'))

In [ ]:
# Visual check: contamination ramp and dimming ramp on one sequence each.
ff = pd.read_csv(f'{FAULT_DIR}/fault_frames.csv')
fig, ax = plt.subplots(2, 4, figsize=(16, 7))
for r, ds in enumerate(['c2c12_fault_contam', 'c2c12_fault_dim']):
    sid = ff[ff.dataset == ds].sequence_id.iloc[0]
    g = ff[ff.sequence_id == sid].sort_values('frame_idx')
    for c, row in enumerate(g.iloc[np.linspace(0, len(g)-1, 4).astype(int)].itertuples()):
        img = cv2.imread(row.png_path, cv2.IMREAD_GRAYSCALE)
        sev = man[(man.fault_sequence_id == sid) & (man.frame_idx == row.frame_idx)].severity.iloc[0]
        ax[r, c].imshow(img, cmap='gray', vmin=0, vmax=255)
        ax[r, c].set_title(f"{ds.split('_')[-1]}  severity={sev:.2f}  mean={img.mean():.0f}", fontsize=9)
        ax[r, c].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Stage everything prepared so far (PNGs + CSVs + manifest; not the raw TIFFs) so a disconnect
# during the GPU pass doesn't mean re-downloading. Skipped if we restored from it.
if not RESTORED:
    !cd / && zip -q -r $PREPARED_ZIP {C2C12_DIR[1:]}/png {C2C12_DIR[1:]}/c2c12_sequences.csv {C2C12_DIR[1:]}/c2c12_frames.csv {C2C12_DIR[1:]}/c2c12_normalization.json {FAULT_DIR[1:]}
    print(f'{PREPARED_ZIP}: {os.path.getsize(PREPARED_ZIP)/1e9:.2f} GB')

## 4 — GPU pass into the existing cache

Every C2C12 and fault frame gets full DINOv2 patch embeddings (Phase A4's per-bin kNN needs patches for
the replay frames). Crops: 4 per fraction instead of nb/02's 8 — replay samples 1–4 per visit, and the FOV
noise model is already fitted (Slice 1b).

In [ ]:
CROP_FRACS = (0.25, 0.5)
CROPS_PER_FRAC = 4

ff = pd.read_csv(f'{FAULT_DIR}/fault_frames.csv')
records, keep_full = [], set()
for r in frames.itertuples():
    records.append(ImageRecord(path=r.png_path, dataset='c2c12', source_path=r.raw_path,
                               sequence_id=r.sequence_id, frame_idx=int(r.frame_idx), timestamp=r.timestamp))
for r in ff.itertuples():
    records.append(ImageRecord(path=r.png_path, dataset=r.dataset, source_path=r.png_path,
                               sequence_id=r.sequence_id, frame_idx=int(r.frame_idx), timestamp=r.timestamp))
keep_full = {image_sha256(r.path) for r in records}
print(len(records), 'images to add |', pd.Series([r.dataset for r in records]).value_counts().to_dict())

In [ ]:
# Budget test: 40 images, stratified by dataset, into a throwaway cache.
random.seed(0)
by_ds = {}
for r in records:
    by_ds.setdefault(r.dataset, []).append(r)
budget = []
for ds, rs in by_ds.items():
    budget += random.sample(rs, max(1, round(40 * len(rs) / len(records))))
BUDGET_DIR = '/content/cache_budget_test_nb03'
!rm -rf $BUDGET_DIR
t0 = time.time()
summ = Cache(BUDGET_DIR).build(budget, models=('seg', 'qc', 'quality', 'dino'), crop_fracs=CROP_FRACS,
                               crops_per_frac=CROPS_PER_FRAC, keep_full_patches_for=keep_full, progress=False)
wall = time.time() - t0
import subprocess
b = int(subprocess.run(['du', '-sb', BUDGET_DIR], capture_output=True, text=True).stdout.split()[0])
print(f'{len(budget)} images in {wall:.0f}s; per-image by model: {summ["per_image_s"]}')
print(f'projected full pass: ~{wall*len(records)/len(budget)/3600:.2f} h, ~{b*len(records)/len(budget)/1e9:.2f} GB added to the Drive cache')

### ⏸ Checkpoint 3 — approve the projected runtime and storage

If it's too long for one Colab session: lower `PER_EXPERIMENT` (e.g. 6 → 18 sequences) or raise `STRIDE`
to 24 (2-hourly frames) and rerun from step 2 — or just run anyway: `Cache.build()` flushes its tables every 100
images, so a disconnect loses at most the last <100 images' rows. Rerun the cell and it skips everything already cached.

In [ ]:
RUN_FULL_PASS = False   # <-- flip after reading the projection above
if not RUN_FULL_PASS:
    raise SystemExit('Stopped at checkpoint 3 — review the budget test, then set RUN_FULL_PASS = True.')

In [ ]:
# One call: build() flushes every 100 images itself. (The old chunk loop was the workaround for the
# flush bug; it re-read every parquet table and rewrote MANIFEST.json per chunk, over Drive FUSE.)
cache = Cache(CACHE_DIR)
t_start = time.time()
cache.build(records, models=('seg', 'qc', 'quality', 'dino'), crop_fracs=CROP_FRACS,
            crops_per_frac=CROPS_PER_FRAC, keep_full_patches_for=keep_full, progress=True)
print(f'full pass done in {(time.time() - t_start)/60:.1f} min')

### 4b — Tile-level DINOv2 embeddings (`crop_spec="qctile"`)

`Cache.build()` embeds the **whole frame**, which DINOv2's processor shrinks to 224 px — a ~4–5x
downscale on 1392x1040 C2C12 frames. Bacterial sprites (2–5 px) vanish, and the embeddings sit at a
different scale from a bank built on 256 px tiles. This adds embeddings of the same 256 px centre tile the
QC classifier sees, at native resolution. **Phase A4 should use `crop_spec="qctile"` for these frames.**
Cheap (one small forward pass per image), resumable, flushes every 200 images.


In [ ]:
from cache_tile_embeddings import build_qctile_embeddings
t0 = time.time()
n_new = build_qctile_embeddings(CACHE_DIR, [r.path for r in records], chunk=200)
print(f'{n_new} qctile embeddings in {(time.time()-t0)/60:.1f} min')


## 5 — Sidecars, slim export, contract check

In [ ]:
# Sidecars the Phase A code needs alongside the cache (export_slim() doesn't know about them).
for dest in (f'{CACHE_DIR}/sidecars', f'{SLIM_DIR}/sidecars'):
    os.makedirs(dest, exist_ok=True)
    for src in (f'{C2C12_DIR}/c2c12_sequences.csv', f'{C2C12_DIR}/c2c12_frames.csv',
                f'{C2C12_DIR}/c2c12_normalization.json', f'{FAULT_DIR}/fault_manifest.parquet',
                f'{FAULT_DIR}/fault_frames.csv', f'{FAULT_DIR}/fault_config.json'):
        shutil.copy2(src, dest)
cache.export_slim(SLIM_DIR)

# export_slim() drops ALL patch embeddings; Phase A4 needs them for the C2C12 + fault frames.
emb_zip = f'{STAGE_DIR}/c2c12_patch_embeddings.zip'
emb_files = [f'{CACHE_DIR}/embeddings/{s}_{spec}.npy' for s in keep_full for spec in ('full', 'qctile')]
emb_files = [p for p in emb_files if os.path.exists(p)]
with open('/content/emb_list.txt', 'w') as f:
    f.write('\n'.join(emb_files))
!zip -q -j $emb_zip -@ < /content/emb_list.txt
print(len(emb_files), 'patch-embedding files ->', emb_zip, f'{os.path.getsize(emb_zip)/1e6:.0f} MB')
print(json.dumps(json.load(open(f'{CACHE_DIR}/MANIFEST.json'))['row_counts'], indent=2))

In [ ]:
# Contract check: the real replay code now runs on a real cached sequence (no fixture).
from culture.replay import build_replay_visits
sid = seqs.sequence_id.iloc[0]
visits = build_replay_visits(Cache(CACHE_DIR), sequence_id=sid, lineage_id='L-smoke', segment_id='S-smoke',
                             flask_id='flask-smoke', seed=0)
print(f'{sid}: {len(visits)} replay visits')
for v in visits:
    print(f"  {v['timestamp'][:16]}  conf {v['confluency_mean']:5.1f} ± {v['confluency_sd']:4.1f}  (n_fov={v['n_fov']})  {v['class_pred']}")

## ⏸ Report back

Copy into `docs/STATUS.md` / the Phase A report: sequences + conditions table, normalization JSON, fault
summary table, budget-test numbers, full-pass wall time, manifest row counts, and the replay smoke output.

**Next, on the Mac:**
1. Pull `cache_slim/` (now with `sidecars/`) and unzip `staged/c2c12_patch_embeddings.zip` into the Mac
   cache's `embeddings/`.
2. **A2 needs one small adapter in `culture/replay.py`:** replay a sequence from an explicit frames table
   (`sidecars/fault_manifest.parquet` rows for one `fault_sequence_id`) instead of only by `sequence_id`
   from `images.parquet`. Fault sequences share their pre-onset frames with the original sequence, and
   `images.parquet` stores one `sequence_id` per image, so they can't be looked up by id alone.
3. Rerun `scripts/backtest_growth.py` on real sequences. `results/growth_backtest.md` currently says it
   must not be quoted until this happens.
4. In A4, build banks and score queries from `crop_spec="qctile"` embeddings for C2C12/fault frames
   (same scale as the 256 px synthetic tiles). Full-frame embeddings can't see small contaminants.
5. Credit C2C12 (CC BY 4.0) in the README's dataset section.